# Step 4 - NIST Statistical Testing on windows


In [ ]:
!git clone https://github.com/stevenang/randomness_testsuite.git
%cd randomness_testsuite

Cloning into 'randomness_testsuite'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 195 (delta 66), reused 58 (delta 58), pack-reused 121 (from 1)
Receiving objects: 100% (195/195), 938.19 KiB | 3.49 MiB/s, done.
Resolving deltas: 100% (103/103), done.
/content/randomness_testsuite


In [ ]:
!ls

ApproximateEntropy.py  LICENSE		    RunTest.py		 test_sqrt2.py
BinaryMatrix.py        Main.py		    Serial.py		 test_sqrt3.py
Complexity.py	       Matrix.py	    Spectral.py		 test_url_01.py
CumulativeSum.py       OLD_Main.py	    TemplateMatching.py  Tools.py
data		       RandomExcursions.py  test_bin_file.py	 Universal.py
FrequencyTest.py       README.md	    test_e.py
GUI.py		       result		    test_pi.py


In [ ]:
import os
import csv
import pandas as pd
from google.colab import files
from IPython.display import display, Javascript
from base64 import b64encode
from FrequencyTest import FrequencyTest
from RunTest import RunTest
from Matrix import Matrix
from Spectral import SpectralTest
from TemplateMatching import TemplateMatching
from Universal import Universal
from Complexity import ComplexityTest
from Serial import Serial
from ApproximateEntropy import ApproximateEntropy
from CumulativeSum import CumulativeSums
from RandomExcursions import RandomExcursions

filename = '/content/qci_qrng_data_121mil.bin'

#Read binary data
with open(filename, "rb") as f:
    raw_bytes = f.read()

binary_data = ''.join(format(byte, '08b') for byte in raw_bytes)
total_bits = len(binary_data)
print(f"Total bits: {total_bits}")
window_size = 1204224
stride = window_size // 2

# Prepare CSV
csv_filename = "/content/nist_window_results.csv"
fields = ["window_index", "start_bit", "end_bit",
          "FrequencyTest", "BlockFrequency", "RunTest", "LongestRun",
          "MatrixRank", "Spectral", "Universal", "ApproxEntropy",
          "LinearComplexity", "Non-overlapping Template Matching Test",
          "Overlappong Template Matching Test",
          "Serial Test", "Cumulative Sums (Forward)",
          "Cumulative Sums (Backward)", "Random Excursion Test",
          "Random Excursion Variant Test"]


def randomExcursion_conclusion(data):
    result = RandomExcursions.random_excursions_test(data)
    passed = len(result) >= 1
    for item in result:
        passed = passed and (item[4] >= 0.01)
    return passed


def randomExcursionVariant_conclusion(data):
    result = RandomExcursions.variant_test(data)
    passed = len(result) >= 1
    for item in result:
        passed = passed and (item[4] >= 0.01)
    return passed


def auto_backup_js(filepath, backup_interval=10, current_window=0):
    if current_window > 0 and current_window % backup_interval == 0:
        print(f"\nAUTO-BACKUP at window {current_window}")
        # Use Javascript to force download
        js_code = f"""
        async function download() {{
            const response = await google.colab.kernel.invokeFunction(
                'notebook.read_file', ['{filepath}'], {{}}
            );
            const data = response.data['application/octet-stream'];
            const blob = new Blob([Uint8Array.from(atob(data), c => c.charCodeAt(0))]);
            const url = URL.createObjectURL(blob);
            const a = document.createElement('a');
            a.href = url;
            a.download = 'nist_window_results.csv';
            document.body.appendChild(a);
            a.click();
            document.body.removeChild(a);
            URL.revokeObjectURL(url);
        }}
        download();
        """
        display(Javascript(js_code))
        print("✅ Backup triggered!\n")


# Calculate which window index this corresponds to
# CUSTOM START POINT - Start from window_index 300
window_index = 1050
start_from_bit = window_index * stride

print(f"🚀 Starting from bit {start_from_bit} (window {window_index})")
print(f"📊 Remaining bits to process: {total_bits - start_from_bit:,}")
print(f"📈 Estimated remaining windows: {(total_bits - start_from_bit - window_size) // stride + 1}\n")

#Resume mechanism - check if CSV exists
if os.path.exists(csv_filename):
    existing = pd.read_csv(csv_filename)
    if not existing.empty:
        # Line 95-102 REPLACE with:
        last_start = existing["start_bit"].max()  # Get LAST window's START bit
        next_start = last_start + stride  # Add stride for 50% overlap

        if next_start > start_from_bit:
            start_from_bit = next_start  # Resume from correct position
            window_index = len(existing)  # Next window number
            write_header = False
            print(f"Found existing CSV! Resuming from bit {start_from_bit}, window {window_index}\n")
        else:
            # Our custom start is ahead, append to CSV
            write_header = False
            print(f"✅ Appending to existing CSV from bit {start_from_bit}\n")
    else:
        write_header = True
else:
    write_header = True

#Run loop with auto-backup
with open(csv_filename, "a", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fields)
    if write_header:
        writer.writeheader()

    for start in range(start_from_bit, total_bits - window_size + 1, stride):
        end = start + window_size
        test_data = binary_data[start:end]

        print(f"✅ Processing window {window_index} — Bits {start:,} to {end:,}")

        try:
            results = {
                "window_index": window_index,
                "start_bit": start,
                "end_bit": end,
                "FrequencyTest": FrequencyTest.monobit_test(test_data),
                "BlockFrequency": FrequencyTest.block_frequency(test_data),
                "RunTest": RunTest.run_test(test_data),
                "LongestRun": RunTest.longest_one_block_test(test_data),
                "MatrixRank": Matrix.binary_matrix_rank_text(test_data),
                "Spectral": SpectralTest.spectral_test(test_data),
                "Universal": Universal.statistical_test(test_data),
                "ApproxEntropy": ApproximateEntropy.approximate_entropy_test(test_data),
                "LinearComplexity": ComplexityTest.linear_complexity_test(test_data),
                "Non-overlapping Template Matching Test": TemplateMatching.non_overlapping_test(test_data),
                "Overlappong Template Matching Test": TemplateMatching.overlapping_patterns(test_data),
                "Serial Test": Serial.serial_test(test_data),
                "Cumulative Sums (Forward)": CumulativeSums.cumulative_sums_test(test_data, 0),
                "Cumulative Sums (Backward)": CumulativeSums.cumulative_sums_test(test_data, 1),
                "Random Excursion Test": randomExcursion_conclusion(test_data),
                "Random Excursion Variant Test": randomExcursionVariant_conclusion(test_data),
            }

            writer.writerow(results)
            csvfile.flush()  # Force write to disk immediately

        except Exception as e:
            print(f"Error processing window {window_index}: {e}")
            # Write row with None values on error
            results = {field: None for field in fields}
            results["window_index"] = window_index
            results["start_bit"] = start
            results["end_bit"] = end
            writer.writerow(results)
            csvfile.flush()

        window_index += 1

        # AUTO-DOWNLOAD every 10 windows (JS version)
        auto_backup_js(csv_filename, backup_interval=10, current_window=window_index)

print(f"\n✅ Complete! Processed {window_index} windows total")
print("⬇️ Downloading final CSV...")
files.download(csv_filename)